In [52]:
from ptflops import get_model_complexity_info
import utils.utils as utils
import yaml

config = utils.parse_config('configs/train_ptb_xl_run_config.yaml', 'config_defaults/train_ptb_xl_config_defaults.yaml')
base_model = utils.get_base_model(config)

In [53]:
import torch
base_model = base_model.to('cuda')
inp_size = (1, 5000, 12)
base_model(torch.rand(inp_size).to('cuda'))

tensor([[ 0.0345, -0.1927,  0.3372, -0.2113, -0.0156]], device='cuda:0',
       grad_fn=<AddmmBackward0>)

In [54]:
def input_constructor(input_res):
    return torch.randn(input_res).to('cuda')

macs, params = get_model_complexity_info(
    base_model, inp_size,
    input_constructor=input_constructor,
    as_strings=True, verbose=True, print_per_layer_stat=True
)
print('FLOPs:', macs)

Net1D(
  30.67 M, 100.000% Params, 1.17 GMac, 99.955% MACs, 
  (first_conv): MyConv1dPadSame(
    12.35 k, 0.040% Params, 30.88 MMac, 2.632% MACs, 
    (conv): Conv1d(12.35 k, 0.040% Params, 30.88 MMac, 2.632% MACs, 12, 64, kernel_size=(16,), stride=(2,))
  )
  (first_bn): BatchNorm1d(128, 0.000% Params, 0.0 Mac, 0.000% MACs, 64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (first_activation): Swish(0, 0.000% Params, 0.0 Mac, 0.000% MACs, )
  (stage_list): ModuleList(
    (0): BasicStage(
      58.69 k, 0.191% Params, 67.29 MMac, 5.736% MACs, 
      (block_list): ModuleList(
        (0): BasicBlock(
          29.34 k, 0.096% Params, 36.32 MMac, 3.096% MACs, 
          (bn1): BatchNorm1d(128, 0.000% Params, 0.0 Mac, 0.000% MACs, 64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (activation1): Swish(0, 0.000% Params, 0.0 Mac, 0.000% MACs, )
          (do1): Dropout(0, 0.000% Params, 0.0 Mac, 0.000% MACs, p=0.5, inplace=False)
          (co

In [55]:
from fvcore.nn import FlopCountAnalysis 

inp = torch.randn(inp_size).to('cuda')   # (batch, seq_len, features)
flops = FlopCountAnalysis(base_model, inp)

flops_val = flops.total()

if flops_val < 1e6:
    print(f"FLOPs: {flops_val:.2f}")
elif flops_val < 1e9:
    print(f"FLOPs: {flops_val/1e6:.2f} MFLOPs")
elif flops_val < 1e12:
    print(f"FLOPs: {flops_val/1e9:.2f} GFLOPs")
else:
    print(f"FLOPs: {flops_val/1e12:.2f} TFLOPs")

Unsupported operator aten::add encountered 42 time(s)
Unsupported operator aten::sub encountered 84 time(s)
Unsupported operator aten::mul encountered 101 time(s)
Unsupported operator aten::pad encountered 71 time(s)
Unsupported operator aten::sigmoid encountered 100 time(s)
Unsupported operator aten::mean encountered 21 time(s)
Unsupported operator aten::max_pool1d encountered 7 time(s)
Unsupported operator aten::add_ encountered 20 time(s)
The following submodules of the model were never called during the trace of the graph. They may be unused, or they were accessed by direct calls to .forward() or via other python methods. In the latter case they will have zeros for statistics, though their statistics will still contribute to their parent calling module.
first_bn, stage_list.0.block_list.0.activation1, stage_list.0.block_list.0.bn1, stage_list.0.block_list.0.bn2, stage_list.0.block_list.0.bn3, stage_list.0.block_list.0.do1, stage_list.0.block_list.0.do2, stage_list.0.block_list.0.do

FLOPs: 1.17 GFLOPs
